# Second model pair: does the finding generalise beyond one model size?

Needs GPU T4 x2, Internet, and the `HF_TOKEN` secret ticked. Trains and evaluates
student_small=Qwen3-1.7B with teacher_8b=Qwen3-8B (~4.7x apart, vs the main pair's Qwen3-14B/
Qwen3-4B at ~3.5x) on the SAME training data (default 8-tool split) and the SAME 121-task
unseen-tool eval set as the main pipeline, so its numbers are directly comparable.
Tokenizer compatibility for this pair is verified (configs/models.yaml's comment) - logit-level
KD is safe.

Conditions: `base`, `sft_only`, `distilled_8b` (KD from teacher_8b), `self_distill_small` (KD
from student_small's own frozen base) - the same three-way core comparison as the main pipeline,
without the noisier early-stopping/label-smoothing controls (scope trimmed for time - see
evaluation/second_pair_worker.py's module docstring). `base` is trained/evaluated once (seed 0
only) since a freshly-initialized LoRA adapter (B=0) is deterministic regardless of seed, same
trick the main pipeline uses.

Two GPU workers train + evaluate their share of the (seed, condition) jobs (this trains, not
just evaluates - more expensive per job than notebooks 11/13). **Stops starting new jobs after
`MAX_MINUTES`** (per worker) and resumes on a later run (finished jobs skipped, checked against
Hugging Face).

In [ ]:
SEEDS = [0, 1, 2]                                            # secondary check: fewer seeds than the main 5
CONDITIONS = ["distilled_8b", "self_distill_small", "sft_only"]
MAX_MINUTES = 150                                             # stop starting new jobs after this long (per worker)

In [ ]:
import base64
import os
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Kaggle secrets (Add-ons -> Secrets), all optional:
#   GH_TOKEN  read access to the GitHub repo, needed only while the repo is private
#   HF_TOKEN  write access to a Hugging Face repo, needed only to resume across sessions
os.environ.setdefault('ADBENCH_HF_REPO', 'NahlaNabil/adbench-run')
os.environ.setdefault('ADBENCH_RUN_TAG', 'v1-fixed')
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GH_TOKEN', 'HF_TOKEN'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            pass
except Exception:
    pass


def git(*args, timeout=600):
    """Run git without ever prompting (a credentials prompt would hang an unattended run for
    hours). Uses GH_TOKEN when set; if that fails (revoked token, or a public repo that needs
    none) it retries once without it."""
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    token = os.environ.get('GH_TOKEN')
    for use_token in ([True, False] if token else [False]):
        cmd = ['git']
        if use_token:
            basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}']
        try:
            subprocess.run(cmd + list(args), check=True, timeout=timeout, env=env)
            return
        except subprocess.CalledProcessError:
            if not use_token:
                raise
            print('git with GH_TOKEN failed; retrying without it.')


def run_module(*args, timeout=4 * 3600):
    """Run `python -m <args>` in a fresh process, print the tail of its output, and raise if it
    fails or exceeds `timeout` seconds (a bare `!` command never stops the notebook)."""
    proc = subprocess.run(
        [sys.executable, '-m', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout
    )
    print(proc.stdout[-20000:])
    proc.check_returncode()


ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    git('clone', 'https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git', REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True, timeout=1800)

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
# re-sync to the latest commit
git('-C', REPO_DIR, 'checkout', '--', '.')
git('-C', REPO_DIR, 'pull')

## 1. Data (the main pipeline's own default split - reused as-is, not the tool-diversity ablation's)

In [ ]:
run_module("adbench.data.prepare", "--config", "configs/data.yaml")

## 2. Check the token can write

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    raise SystemExit("HF_TOKEN is not set: tick it under Add-ons -> Secrets for this notebook, then run again.")
os.environ["ADBENCH_RUN_TAG"] = "v2-seed0"
from adbench import pipeline_state as ps
ps.check_upload()      # raises if the token cannot write

## 3. Train + evaluate

**Two phases, not one** -- same reason as notebook 14's tool-diversity ablation (see there for
the full explanation): a condition that needs a teacher (`distilled_8b`, `self_distill_small`)
loads two models at once, and that only splits across both physical GPUs automatically when the
process can see both -- pinning a worker to a single GPU (right for `sft_only`/`base`) makes
Unsloth try to fit teacher_8b (~8B) AND student_small (~1.7B) on ONE T4 together, which does not
fit. So teacher-needing conditions run sequentially, one process, both GPUs visible; teacher-free
conditions run as two GPU-pinned workers in parallel, as before. Which is which is read from
configs/experiment.yaml (`resolve_training_config(...).kd.kd_weight > 0`).

In [ ]:
import subprocess
import sys
import threading

from adbench.training.train import load_experiment_config as _load_exp_cfg, resolve_training_config as _resolve_cfg

_exp_cfg = _load_exp_cfg("configs/experiment.yaml")
ALL_CONDITIONS_HERE = ["base"] + CONDITIONS
TEACHER_CONDITIONS = {c for c in ALL_CONDITIONS_HERE if _resolve_cfg(_exp_cfg, c).kd.kd_weight > 0}
NO_TEACHER_CONDITIONS = set(ALL_CONDITIONS_HERE) - TEACHER_CONDITIONS
print("teacher conditions (sequential, both GPUs):", TEACHER_CONDITIONS)
print("no-teacher conditions (parallel, one GPU each):", NO_TEACHER_CONDITIONS)


def pump(proc, name):
    for line in proc.stdout:
        if "Loading weights" in line or "it/s]" in line:
            continue
        print(f"[{name}] {line.rstrip()}", flush=True)


def run_worker(jobs, env_overrides, work_dir_tag):
    env = {**os.environ, **env_overrides, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        [sys.executable, "-m", "adbench.evaluation.second_pair_worker", "--jobs", ",".join(jobs),
         "--work-dir", f"/kaggle/working/pair2_{work_dir_tag}", "--max-minutes", str(MAX_MINUTES)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
    )
    thread = threading.Thread(target=pump, args=(proc, work_dir_tag), daemon=True)
    thread.start()
    return proc, thread


def make_jobs(conditions):
    jobs = []
    if "base" in conditions:
        jobs.append("0:base")
    jobs += [f"{seed}:{cond}" for cond in conditions if cond != "base" for seed in SEEDS]
    return jobs


# --- Phase A: no-teacher conditions, two GPU-pinned workers in parallel ---
no_teacher_jobs = make_jobs(NO_TEACHER_CONDITIONS)
if no_teacher_jobs:
    print(f"### Phase A: {len(no_teacher_jobs)} no-teacher jobs, two parallel workers ###")
    workers = []
    for gpu in (0, 1):
        share = no_teacher_jobs[gpu::2]
        if not share:
            continue
        workers.append(run_worker(share, {"CUDA_VISIBLE_DEVICES": str(gpu)}, f"phaseA_gpu{gpu}"))
    for proc, thread in workers:
        proc.wait()
        thread.join(timeout=30)
    codes = [proc.returncode for proc, _ in workers]
    print("Phase A exit codes:", codes)
    if any(codes):
        raise RuntimeError("a Phase A worker failed; see the log above")

# --- Phase B: teacher conditions, one process, both GPUs visible, sequential ---
teacher_jobs = make_jobs(TEACHER_CONDITIONS)
if teacher_jobs:
    print(f"### Phase B: {len(teacher_jobs)} teacher jobs, sequential, both GPUs ###")
    proc, thread = run_worker(teacher_jobs, {}, "phaseB")
    proc.wait()
    thread.join(timeout=30)
    print("Phase B exit code:", proc.returncode)
    if proc.returncode:
        raise RuntimeError("the Phase B worker failed; see the log above")

## Summary of what was evaluated

In [ ]:
import glob
import json

import pandas as pd

rows = []
for path in glob.glob("/kaggle/working/pair2_phase*/results/*.json"):
    rows += json.load(open(path, encoding="utf-8"))
if rows:
    df = pd.DataFrame(rows)
    table = df.groupby(["condition", "seed", "chain_length"]).success.mean().unstack("chain_length").round(3)
    print(table)
else:
    print("no result files in this session (everything was already on Hugging Face)")